# CNN–Transformer Image Captioning: Inference Pipeline

This notebook demonstrates image caption generation using a frozen **VGG16 CNN encoder** followed by a **Transformer encoder–decoder** model. The generated English caption is then processed for keyword-based hashtag generation and translated into selected regional languages using mBART-50.

## Reproducibility

This notebook assumes the following local project structure:

```text
project/
├── data/
│   ├── captions.txt
│   └── Images/
├── artifacts/
│   ├── tokenizer_vocab.json
│   └── transformer_saved_model_full.keras
└── test_images/
    └── example.jpg
```

The dataset and trained model artifacts are **not included in the repository** when their size/licensing makes redistribution inappropriate. Obtain them separately and place them in the paths above.


# **Inference on actual image**

### Image caption and hashtags generation using CNN + Transformer, and then converting it in Regional language

In [ ]:
import tensorflow as tf
import os
import json
import pandas as pd
import re
import numpy as np
import time
import matplotlib.pyplot as plt
import collections
import random
import requests
from math import sqrt
from PIL import Image
from tqdm.auto import tqdm

## Dataset and Local Paths

In [ ]:
from pathlib import Path

# Expected local project structure:
# data/
# ├── captions.txt
# └── Images/
#
# Update DATA_DIR if your Flickr8k dataset is stored elsewhere.
DATA_DIR = Path("data")
CAPTIONS_PATH = DATA_DIR / "captions.txt"
IMAGES_DIR = DATA_DIR / "Images"

captions = pd.read_csv(CAPTIONS_PATH)
captions['image'] = captions['image'].apply(lambda x: str(IMAGES_DIR / x))
captions.head()


## Caption Preprocessing

In [ ]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub('\s+', ' ', text)
    text = text.strip()
    text = '[start] ' + text + ' [end]'
    return text

In [ ]:
captions['caption'] = captions['caption'].apply(preprocess)
captions.head()

## Model and Training Configuration

In [ ]:
MAX_LENGTH = 40
VOCABULARY_SIZE = 10000
BATCH_SIZE = 16
BUFFER_SIZE = 1000
EMBEDDING_DIM = 512
UNITS = 512

## Caption Tokenization

In [ ]:
tokenizer = tf.keras.layers.TextVectorization(
    max_tokens=VOCABULARY_SIZE,
    standardize=None,
    output_sequence_length=MAX_LENGTH)

tokenizer.adapt(captions['caption'])

In [ ]:
word2idx = tf.keras.layers.StringLookup(
    mask_token="",
    vocabulary=tokenizer.get_vocabulary())

idx2word = tf.keras.layers.StringLookup(
    mask_token="",
    vocabulary=tokenizer.get_vocabulary(),
    invert=True)

## Train–Validation Split

In [ ]:
img_to_cap_vector = collections.defaultdict(list)
for img, cap in zip(captions['image'], captions['caption']):
    img_to_cap_vector[img].append(cap)

img_keys = list(img_to_cap_vector.keys())
random.shuffle(img_keys)

slice_index = int(len(img_keys)*0.8)
img_name_train_keys, img_name_val_keys = (img_keys[:slice_index],
                                          img_keys[slice_index:])

train_imgs = []
train_captions = []
for imgt in img_name_train_keys:
    capt_len = len(img_to_cap_vector[imgt])
    train_imgs.extend([imgt] * capt_len)
    train_captions.extend(img_to_cap_vector[imgt])

val_imgs = []
val_captions = []
for imgv in img_name_val_keys:
    capv_len = len(img_to_cap_vector[imgv])
    val_imgs.extend([imgv] * capv_len)
    val_captions.extend(img_to_cap_vector[imgv])

In [ ]:
len(train_imgs), len(train_captions), len(val_imgs), len(val_captions)

## Image and Caption Loading

In [ ]:
def load_data(img_path, caption):
    img = tf.io.read_file(img_path)
    img = tf.io.decode_jpeg(img, channels=3)
    img = tf.keras.layers.Resizing(299, 299)(img)
    img = img / 255.
    caption = tokenizer(caption)
    return img, caption

In [ ]:
train_dataset = tf.data.Dataset.from_tensor_slices(
    (train_imgs, train_captions))

train_dataset = train_dataset.map(
    load_data, num_parallel_calls=tf.data.AUTOTUNE
    ).shuffle(BUFFER_SIZE).batch(BATCH_SIZE)

val_dataset = tf.data.Dataset.from_tensor_slices(
    (val_imgs, val_captions))

val_dataset = val_dataset.map(
    load_data, num_parallel_calls=tf.data.AUTOTUNE
    ).shuffle(BUFFER_SIZE).batch(BATCH_SIZE)

## Image Augmentation

In [ ]:
image_augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.2),
        tf.keras.layers.RandomContrast(0.3),
    ]
)

## CNN Image Encoder

In [ ]:
import tensorflow as tf

def CNN_Encoder():
    vgg16 = tf.keras.applications.VGG16(
        include_top=False,
        weights='imagenet'
    )

    vgg16.trainable = False

    output = vgg16.output
    output = tf.keras.layers.Reshape(
        (-1, output.shape[-1])
    )(output)

    cnn_model = tf.keras.models.Model(vgg16.input, output)

    return cnn_model


## Transformer Encoder

In [ ]:
class TransformerEncoderLayer(tf.keras.layers.Layer):

    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.layer_norm_1 = tf.keras.layers.LayerNormalization()
        self.layer_norm_2 = tf.keras.layers.LayerNormalization()
        self.attention = tf.keras.layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim)
        self.dense = tf.keras.layers.Dense(embed_dim, activation="relu")


    def call(self, x, training):
        x = self.layer_norm_1(x)
        x = self.dense(x)

        attn_output = self.attention(
            query=x,
            value=x,
            key=x,
            attention_mask=None,
            training=training
        )

        x = self.layer_norm_2(x + attn_output)
        return x

## Token and Positional Embeddings

In [ ]:
class Embeddings(tf.keras.layers.Layer):

    def __init__(self, vocab_size, embed_dim, max_len):
        super().__init__()
        self.token_embeddings = tf.keras.layers.Embedding(
            vocab_size, embed_dim)
        self.position_embeddings = tf.keras.layers.Embedding(
            max_len, embed_dim, input_shape=(None, max_len))


    def call(self, input_ids):
        length = tf.shape(input_ids)[-1]
        position_ids = tf.range(start=0, limit=length, delta=1)
        position_ids = tf.expand_dims(position_ids, axis=0)

        token_embeddings = self.token_embeddings(input_ids)
        position_embeddings = self.position_embeddings(position_ids)

        return token_embeddings + position_embeddings

In [ ]:
Embeddings(tokenizer.vocabulary_size(), EMBEDDING_DIM, MAX_LENGTH)(next(iter(train_dataset))[1]).shape

## Transformer Decoder

In [ ]:
class TransformerDecoderLayer(tf.keras.layers.Layer):

    def __init__(self, embed_dim, units, num_heads):
        super().__init__()
        self.embedding = Embeddings(
            tokenizer.vocabulary_size(), embed_dim, MAX_LENGTH)

        self.attention_1 = tf.keras.layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim, dropout=0.1
        )
        self.attention_2 = tf.keras.layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim, dropout=0.1
        )

        self.layernorm_1 = tf.keras.layers.LayerNormalization()
        self.layernorm_2 = tf.keras.layers.LayerNormalization()
        self.layernorm_3 = tf.keras.layers.LayerNormalization()

        self.ffn_layer_1 = tf.keras.layers.Dense(units, activation="relu")
        self.ffn_layer_2 = tf.keras.layers.Dense(embed_dim)

        self.out = tf.keras.layers.Dense(tokenizer.vocabulary_size(), activation="softmax")

        self.dropout_1 = tf.keras.layers.Dropout(0.3)
        self.dropout_2 = tf.keras.layers.Dropout(0.5)


    def call(self, input_ids, encoder_output, training, mask=None):
        embeddings = self.embedding(input_ids)

        combined_mask = None
        padding_mask = None

        if mask is not None:
            causal_mask = self.get_causal_attention_mask(embeddings)
            padding_mask = tf.cast(mask[:, :, tf.newaxis], dtype=tf.int32)
            combined_mask = tf.cast(mask[:, tf.newaxis, :], dtype=tf.int32)
            combined_mask = tf.minimum(combined_mask, causal_mask)

        attn_output_1 = self.attention_1(
            query=embeddings,
            value=embeddings,
            key=embeddings,
            attention_mask=combined_mask,
            training=training
        )

        out_1 = self.layernorm_1(embeddings + attn_output_1)

        attn_output_2 = self.attention_2(
            query=out_1,
            value=encoder_output,
            key=encoder_output,
            attention_mask=padding_mask,
            training=training
        )

        out_2 = self.layernorm_2(out_1 + attn_output_2)

        ffn_out = self.ffn_layer_1(out_2)
        ffn_out = self.dropout_1(ffn_out, training=training)
        ffn_out = self.ffn_layer_2(ffn_out)

        ffn_out = self.layernorm_3(ffn_out + out_2)
        ffn_out = self.dropout_2(ffn_out, training=training)
        preds = self.out(ffn_out)
        return preds


    def get_causal_attention_mask(self, inputs):
        input_shape = tf.shape(inputs)
        batch_size, sequence_length = input_shape[0], input_shape[1]
        i = tf.range(sequence_length)[:, tf.newaxis]
        j = tf.range(sequence_length)
        mask = tf.cast(i >= j, dtype="int32")
        mask = tf.reshape(mask, (1, input_shape[1], input_shape[1]))
        mult = tf.concat(
            [tf.expand_dims(batch_size, -1), tf.constant([1, 1], dtype=tf.int32)],
            axis=0
        )
        return tf.tile(mask, mult)

## Image Captioning Model

In [ ]:
class ImageCaptioningModel(tf.keras.Model):

    def __init__(self, cnn_model, encoder, decoder, image_aug=None):
        super().__init__()
        self.cnn_model = cnn_model
        self.encoder = encoder
        self.decoder = decoder
        self.image_aug = image_aug
        self.loss_tracker = tf.keras.metrics.Mean(name="loss")
        self.acc_tracker = tf.keras.metrics.Mean(name="accuracy")


    def calculate_loss(self, y_true, y_pred, mask):
        loss = self.loss(y_true, y_pred)
        mask = tf.cast(mask, dtype=loss.dtype)
        loss *= mask
        return tf.reduce_sum(loss) / tf.reduce_sum(mask)


    def calculate_accuracy(self, y_true, y_pred, mask):
        accuracy = tf.equal(y_true, tf.argmax(y_pred, axis=2))
        accuracy = tf.math.logical_and(mask, accuracy)
        accuracy = tf.cast(accuracy, dtype=tf.float32)
        mask = tf.cast(mask, dtype=tf.float32)
        return tf.reduce_sum(accuracy) / tf.reduce_sum(mask)


    def compute_loss_and_acc(self, img_embed, captions, training=True):
        encoder_output = self.encoder(img_embed, training=True)
        y_input = captions[:, :-1]
        y_true = captions[:, 1:]
        mask = (y_true != 0)
        y_pred = self.decoder(
            y_input, encoder_output, training=True, mask=mask
        )
        loss = self.calculate_loss(y_true, y_pred, mask)
        acc = self.calculate_accuracy(y_true, y_pred, mask)
        return loss, acc


    def train_step(self, batch):
        imgs, captions = batch

        if self.image_aug:
            imgs = self.image_aug(imgs)

        img_embed = self.cnn_model(imgs)

        with tf.GradientTape() as tape:
            loss, acc = self.compute_loss_and_acc(
                img_embed, captions
            )

        train_vars = (
            self.encoder.trainable_variables + self.decoder.trainable_variables
        )
        grads = tape.gradient(loss, train_vars)
        self.optimizer.apply_gradients(zip(grads, train_vars))
        self.loss_tracker.update_state(loss)
        self.acc_tracker.update_state(acc)

        return {"loss": self.loss_tracker.result(), "acc": self.acc_tracker.result()}


    def test_step(self, batch):
        imgs, captions = batch

        img_embed = self.cnn_model(imgs)

        loss, acc = self.compute_loss_and_acc(
            img_embed, captions, training=False
        )

        self.loss_tracker.update_state(loss)
        self.acc_tracker.update_state(acc)

        return {"loss": self.loss_tracker.result(), "acc": self.acc_tracker.result()}

    @property
    def metrics(self):
        return [self.loss_tracker, self.acc_tracker]

In [ ]:
VOCABULARY_SIZE = 10000
BATCH_SIZE = 16
BUFFER_SIZE = 1000
EMBEDDING_DIM = 512
UNITS = 512

# Saved training artifacts required for inference.
ARTIFACT_DIR = Path("artifacts")
VOCAB_PATH = ARTIFACT_DIR / "tokenizer_vocab.json"
MODEL_WEIGHTS_PATH = ARTIFACT_DIR / "transformer_saved_model_full.keras"


## Build Inference Model

In [ ]:
vocab_size = len(saved_vocab)
print("vocab_size:", vocab_size)

encoder = TransformerEncoderLayer(EMBEDDING_DIM, 1)
decoder = TransformerDecoderLayer(EMBEDDING_DIM, UNITS, 8)
cnn_model = CNN_Encoder()

caption_model_test = ImageCaptioningModel(cnn_model=cnn_model, encoder=encoder, decoder=decoder, image_aug=None)


In [ ]:

class ImageCaptioningModel(tf.keras.Model):
    def __init__(self, cnn_model=None, encoder=None, decoder=None, image_aug=None, **kwargs):
        super().__init__(**kwargs)
        self.cnn_model = cnn_model
        self.encoder = encoder
        self.decoder = decoder
        self.image_aug = image_aug

    def call(self, imgs, captions=None, training=False):
        img_embed = self.cnn_model(imgs, training=training)
        enc = self.encoder(img_embed, training=training)
        if captions is None:
            return enc
        mask = tf.cast(captions != 0, tf.int32)
        logits = self.decoder(captions, enc, training=training, mask=mask)
        return logits

caption_model_test = ImageCaptioningModel(cnn_model=cnn_model, encoder=encoder, decoder=decoder, image_aug=None)
_dummy_img = tf.zeros((1,299,299,3), dtype=tf.float32)
_dummy_caption = tf.zeros((1, MAX_LENGTH), dtype=tf.int32)
_ = caption_model_test(_dummy_img, _dummy_caption, training=False)


In [ ]:
if not MODEL_WEIGHTS_PATH.exists():
    raise FileNotFoundError(
        f"Model weights not found at {MODEL_WEIGHTS_PATH}. "
        "Place the trained model artifact in the artifacts/ directory before running inference."
    )

caption_model_test.load_weights(MODEL_WEIGHTS_PATH)
print("Model weights loaded successfully.")


## Caption Generation

In [ ]:
vocab = tokenizer.get_vocabulary()

def idx_to_token(idx):
    return vocab[int(idx)]

def load_image_from_path(img_path):
    img = tf.io.read_file(img_path)
    img = tf.io.decode_jpeg(img, channels=3)
    img = tf.cast(img, tf.float32)
    img = tf.keras.layers.Resizing(299, 299)(img)
    img = img / 255.0
    return img

def generate_caption_inference(img_path):
    img = load_image_from_path(img_path)
    img = tf.expand_dims(img, axis=0)
    img_embed = caption_model_test.cnn_model(img)
    img_encoded = caption_model_test.encoder(img_embed, training=False)

    y_inp = "[start]"
    for i in range(MAX_LENGTH-1):
        tokenized = tokenizer([y_inp])[:, :-1]
        mask = tf.cast(tokenized != 0, tf.int32)
        preds = caption_model_test.decoder(tokenized, img_encoded, training=False, mask=mask)

        pred_idx = np.argmax(preds[0, i, :])
        pred_word = idx_to_token(pred_idx)

        if pred_word == "[end]":
            break

        if pred_word and pred_word != "":
            y_inp += " " + pred_word

    return y_inp.replace("[start] ", "").strip()


## Inference on a Test Image

In [ ]:
# Provide a local test image path.
# Example: TEST_IMAGE_PATH = Path("test_images/example.jpg")
TEST_IMAGE_PATH = Path("test_images/example.jpg")

if not TEST_IMAGE_PATH.exists():
    raise FileNotFoundError(
        f"Test image not found at {TEST_IMAGE_PATH}. "
        "Add a test image to test_images/ or update TEST_IMAGE_PATH."
    )

im = Image.open(TEST_IMAGE_PATH).convert("RGB")

plt.figure(figsize=(10/2.54, 10/2.54))
plt.imshow(im)
plt.axis('off')
plt.show()

pred_caption = generate_caption_inference(str(TEST_IMAGE_PATH))
print("Predicted Caption:", pred_caption)


## Inference on a Validation Image

In [ ]:
# Example inference using an image from the validation split.
src_path = img_name_val_keys[9]
im = Image.open(src_path).convert("RGB")

plt.figure(figsize=(10/2.54, 10/2.54))
plt.imshow(im)
plt.axis('off')
plt.title("Validation Image")
plt.show()

pred_caption = generate_caption_inference(str(src_path))
print(f"Predicted Caption: {pred_caption}\n")


## Hashtag Generation and Regional-Language Translation

In [ ]:
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast
import nltk

# Download the NLTK resources required for keyword extraction.
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

def extract_keywords(content):
    tokens = nltk.word_tokenize(content)
    tags = nltk.pos_tag(tokens)
    keywords = [
        word for (word, tag) in tags
        if tag.startswith('NN') or tag == 'JJ' or tag == 'NNP'
    ]
    return [word for word in keywords if len(word.strip()) > 2]

def generate_hashtags(keywords):
    return ['#' + word.strip().replace(' ', '') for word in keywords]

translation_model = MBartForConditionalGeneration.from_pretrained(
    "facebook/mbart-large-50-one-to-many-mmt"
)
translation_tokenizer = MBart50TokenizerFast.from_pretrained(
    "facebook/mbart-large-50-one-to-many-mmt",
    src_lang="en_XX"
)

article_en = pred_caption

langs = ["hi_IN", "te_IN", "bn_IN", "ta_IN"]
translations_dict = {}

en_keywords = extract_keywords(article_en)
en_hashtags = generate_hashtags(en_keywords)
translations_dict["en_XX"] = {
    "caption": article_en,
    "hashtag": ", ".join(en_hashtags)
}

for lang in langs:
    model_inputs = translation_tokenizer(article_en, return_tensors="pt")
    generated_tokens = translation_model.generate(
        **model_inputs,
        forced_bos_token_id=translation_tokenizer.lang_code_to_id[lang]
    )

    caption = translation_tokenizer.batch_decode(
        generated_tokens,
        skip_special_tokens=True
    )[0]

    keywords = extract_keywords(caption)
    hashtags = generate_hashtags(keywords)

    translations_dict[lang] = {
        "caption": caption,
        "hashtag": ", ".join(hashtags)
    }


In [ ]:

print(json.dumps(translations_dict, indent=2, ensure_ascii=False))